# Train YOLOv8n Fall Detection tren Kaggle

Notebook nay dung de train YOLOv8n tren Kaggle va lay file `best.pt`.

Dataset can add vao Kaggle Input:

```text
yolo_coffee_room/
??? data.yaml
??? images/
?   ??? train/
?   ??? val/
??? labels/
    ??? train/
    ??? val/
```

Output can tai ve sau train:

```text
/kaggle/working/best_model.zip
```

## 1. Kiem tra moi truong va GPU

Neu GPU chua bat, vao `Settings -> Accelerator -> GPU T4 x2`.

In [ ]:
!pwd
!nvidia-smi || true

## 2. Cai thu vien YOLOv8

In [ ]:
!pip install ultralytics -q

## 3. Tim dataset trong `/kaggle/input`

Cell nay tu dong tim file `data.yaml`, nen khong can biet chinh xac Kaggle mount dataset o duong dan nao.

In [ ]:
from pathlib import Path

input_root = Path('/kaggle/input')
yaml_files = sorted(input_root.rglob('data.yaml'))

print('So file data.yaml tim thay:', len(yaml_files))
for i, path in enumerate(yaml_files):
    print(f'{i}: {path}')

if not yaml_files:
    raise FileNotFoundError('Khong tim thay data.yaml trong /kaggle/input. Hay Add Input dataset truoc.')

DATA_YAML_INPUT = yaml_files[0]
DATASET_INPUT_DIR = DATA_YAML_INPUT.parent

print('DATA_YAML_INPUT =', DATA_YAML_INPUT)
print('DATASET_INPUT_DIR =', DATASET_INPUT_DIR)

## 4. Copy dataset sang `/kaggle/working`

`/kaggle/input` chi doc duoc, nen can copy sang `/kaggle/working` de sua `data.yaml`.

In [ ]:
import shutil
from pathlib import Path

DATASET_WORK_DIR = Path('/kaggle/working/yolo_coffee_room')

if DATASET_WORK_DIR.exists():
    shutil.rmtree(DATASET_WORK_DIR)

shutil.copytree(DATASET_INPUT_DIR, DATASET_WORK_DIR)

print('Copied dataset to:', DATASET_WORK_DIR)

## 5. Sua `data.yaml` cho duong dan Kaggle

In [ ]:
DATA_YAML = DATASET_WORK_DIR / 'data.yaml'

text = DATA_YAML.read_text(encoding='utf-8')

# Dataset tao tren may local/Colab co the dang tro ve /content.
text = text.replace('/content/yolo_coffee_room', str(DATASET_WORK_DIR))

# Neu data.yaml co dong path khac, ghi de lai cho chac chan.
lines = text.splitlines()
new_lines = []
path_written = False

for line in lines:
    if line.strip().startswith('path:'):
        new_lines.append(f'path: {DATASET_WORK_DIR}')
        path_written = True
    else:
        new_lines.append(line)

if not path_written:
    new_lines.insert(0, f'path: {DATASET_WORK_DIR}')

DATA_YAML.write_text(chr(10).join(new_lines) + chr(10), encoding='utf-8')

print(DATA_YAML)
print(DATA_YAML.read_text(encoding='utf-8'))


## 6. Kiem tra so anh va label

In [ ]:
train_images = list((DATASET_WORK_DIR / 'images' / 'train').glob('*'))
val_images = list((DATASET_WORK_DIR / 'images' / 'val').glob('*'))
train_labels = list((DATASET_WORK_DIR / 'labels' / 'train').glob('*.txt'))
val_labels = list((DATASET_WORK_DIR / 'labels' / 'val').glob('*.txt'))

print('Train images:', len(train_images))
print('Train labels:', len(train_labels))
print('Val images:', len(val_images))
print('Val labels:', len(val_labels))

if len(train_images) == 0 or len(val_images) == 0:
    raise RuntimeError('Dataset chua dung cau truc images/train va images/val')

## 7. Train YOLOv8n

- De test nhanh: dat `EPOCHS = 5` hoac `10`.
- De train tot hon: dat `EPOCHS = 50`.

In [ ]:
from ultralytics import YOLO

EPOCHS = 50
IMG_SIZE = 640
BATCH_SIZE = 16

model = YOLO('yolov8n.pt')

results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=0,
    name='fall_detection_yolov8n'
)

## 8. Tim `best.pt` sau train

In [ ]:
from pathlib import Path

best_files = sorted(Path('/kaggle/working').rglob('best.pt'))

print('So file best.pt tim thay:', len(best_files))
for p in best_files:
    print(p)

if not best_files:
    raise FileNotFoundError('Khong tim thay best.pt. Kiem tra lai cell train co chay xong khong.')

BEST_PT = best_files[0]
print('BEST_PT =', BEST_PT)

## 9. Copy va zip `best.pt` de tai ve

File can tai ve la:

```text
/kaggle/working/best_model.zip
```

In [ ]:
import shutil
import zipfile
from pathlib import Path

shutil.copy(BEST_PT, '/kaggle/working/best.pt')
print('Copied to /kaggle/working/best.pt')

zip_path = Path('/kaggle/working/best_model.zip')
if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write('/kaggle/working/best.pt', arcname='best.pt')

print('Created:', zip_path)
print('best.pt size:', Path('/kaggle/working/best.pt').stat().st_size)
print('zip size:', zip_path.stat().st_size)


## 10. Luu output tren Kaggle

Sau khi tao xong `best_model.zip`:

1. Bam `Save Version`.
2. Chon `Save & Run All` neu muon Kaggle luu output chinh thuc.
3. Hoac tai truc tiep `best_model.zip` trong panel Output neu file da hien.

Sau khi tai ve may, giai nen va dat:

```text
C:\Users\ASUS-PRO\Documents\lezzi_datasets\models\best.pt
```

# Tuy chon: validate lai model tot nhat

In [ ]:
from ultralytics import YOLO

best_model = YOLO(str(BEST_PT))
metrics = best_model.val(data=str(DATA_YAML))